# Notebook 4: BART
Evaluates zero-shot bart-mnli on goemotions and exports shared-eval outputs.


In [ ]:
# install zero-shot dependencies and import runtime libraries
!pip install -q -U "transformers>=4.46,<5" "datasets" "accelerate>=0.30"

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import pipeline

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 61.6 MB/s eta 0:00:00
Mounted at /content/drive
PyTorch: 2.10.0+cu128
GPU available: True
GPU: NVIDIA H100 80GB HBM3


In [2]:
# clone repo and enter notebooks directory
!git clone https://github.com/phisomni-edu/cs4120-project /content/cs4120-project
%cd /content/cs4120-project/notebooks

Cloning into '/content/cs4120-project'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 305 (delta 71), reused 65 (delta 42), pack-reused 200 (from 1)
Receiving objects: 100% (305/305), 9.01 MiB | 16.13 MiB/s, done.
Resolving deltas: 100% (181/181), done.
/content/cs4120-project/notebooks


In [ ]:
# resolve repo/data/results directories
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate repo root with src/ and data/ directories")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / "data"
RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Repo root: /content/cs4120-project
Data dir: /content/cs4120-project/data
Results dir: /content/cs4120-project/results


In [ ]:
# parse labels and prepare shared-evaluation targets
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from src.evaluate import evaluate_run

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()
    def _parse_one(value):
        if isinstance(value, list) or isinstance(value, np.ndarray):
            return [int(x) for x in value]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if not body: return []
                toks = [t.strip() for t in body.split(',') if t.strip()] if ',' in body else [t for t in body.split() if t]
                return [int(t) for t in toks]
            return [int(x) for x in ast.literal_eval(s)]
        return value
    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _choose_text_col(df):
    for col in ["text_clean", "text"]:
        if col in df.columns: return col
    raise ValueError("No text column found.")

def _get_label_names(df):
    try:
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
        return ds["train"].features["labels"].feature.names
    except Exception:
        max_id = max(max(labels) if labels else -1 for labels in df["labels"])
        return [f"label_{i}" for i in range(max_id + 1)]

test_df = _parse_labels_column(pd.read_csv(DATA_DIR / "test.csv"))
text_col = _choose_text_col(test_df)
train_df = _parse_labels_column(pd.read_csv(DATA_DIR / "train.csv")) # Just for label space
label_names = _get_label_names(train_df)

mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
mlb.fit([list(range(len(label_names)))])
y_test = mlb.transform(test_df["labels"])

print(f"Loaded {len(test_df)} test samples. Labels: {len(label_names)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Loaded 5427 test samples. Labels: 28


In [ ]:
# reduce logging noise and initialize zero-shot pipeline
import logging
import warnings
logging.getLogger("transformers").setLevel(logging.CRITICAL)
logging.getLogger("transformers.modeling_utils").setLevel(logging.CRITICAL)
logging.getLogger("transformers.configuration_utils").setLevel(logging.CRITICAL)
logging.disable(logging.ERROR)
warnings.filterwarnings("ignore")

from transformers import pipeline
from transformers.models.bart.configuration_bart import BartConfig

MODEL_NAME = "facebook/bart-large-mnli"
device = 0 if torch.cuda.is_available() else -1

print(f"Loading {MODEL_NAME} on device {device}...")

if not hasattr(BartConfig, "forced_bos_token_id"):
    BartConfig.forced_bos_token_id = None

classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=device,
)

print("Pipeline loaded successfully.")

Loading facebook/bart-large-mnli on device 0...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Pipeline loaded successfully.


In [ ]:
# larger batches improve throughput for zero-shot pipeline inference
BATCH_SIZE = 16
texts = test_df[text_col].fillna("").astype(str).tolist()

print("Starting inference (this may take a while)...")
predictions = classifier(
    texts,
    candidate_labels=label_names,
    multi_label=True,
    batch_size=BATCH_SIZE
)

# fixed threshold converts zero-shot scores into binary labels
THRESHOLD = 0.5
y_pred = np.zeros_like(y_test)

# convert per-example label scores into multilabel indicator rows
for i, pred in enumerate(predictions):
    for label, score in zip(pred['labels'], pred['scores']):
        if score >= THRESHOLD:
            label_idx = label_names.index(label)
            y_pred[i, label_idx] = 1

print("Inference complete.")

Starting inference (this may take a while)...
Inference complete.


In [ ]:
# repeat scoring across seeds for consistent result-table schema
for seed in [7, 21, 42]:
    ev = evaluate_run(
        method="zero_shot_bart",
        data_fraction=0.0,
        seed=seed,
        label_names=label_names,
        y_true=y_test,
        y_pred=y_pred,
    )

    ev["overall"].to_csv(RESULTS_DIR / f"zero_shot_overall_seed{seed}.csv", index=False)
    ev["per_class"].to_csv(RESULTS_DIR / f"zero_shot_per_class_seed{seed}.csv", index=False)

    readme_df = ev["per_class"][["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
    readme_df.to_csv(RESULTS_DIR / f"zero_shot_results_seed{seed}.csv", index=False)

    print(f"Seed {seed} saved.")
    display(ev["overall"])

Seed 7 saved.


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
0,zero_shot_bart,0.0,7,0.007371,0.19028,0.171349,0.199163


Seed 21 saved.


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
0,zero_shot_bart,0.0,21,0.007371,0.19028,0.171349,0.199163


Seed 42 saved.


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
0,zero_shot_bart,0.0,42,0.007371,0.19028,0.171349,0.199163
